In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# leemos los resultados de Distance based
distance = pd.read_csv(
    "resultado_distance_based.csv"
)

# leemos los resultados de DBSCAN
dbscan = pd.read_csv(
    "resultado_dbscan.csv"
)

# leemos CSV de zahn
zahn = pd.read_csv(
    "zahn.csv",
    sep="\t",
    header=None,
    names=["X", "Y", "Clase"]
)

In [3]:
# Guardamos todos los resultados en un solo DataFrame para compararlos
comparacion = pd.merge(
    distance,
    dbscan,
    on="indice"
)

comparacion.head()

,indice,anomalia_distance,label_eps_0.20,ruido_eps_0.20,label_eps_0.30,ruido_eps_0.30,label_eps_0.40,ruido_eps_0.40
0,0,True,-1,True,-1,True,0,False
1,1,True,-1,True,-1,True,0,False
2,2,True,-1,True,0,False,0,False
3,3,True,-1,True,-1,True,0,False
4,4,True,-1,True,-1,True,0,False


# Calculo de diferencias mediante Jaccard.

Para dos conjuntos:
$$
A=\text{anomalías Distance-Based}
$$
$$
B=\text{ruido DBSCAN}
$$

tenemos:
$$
J(A,B)=
\frac{|A\cap B|}
{|A\cup B|}
$$


In [4]:
def jaccard(a, b):

    A = set(np.where(a)[0])
    B = set(np.where(b)[0])

    interseccion = A & B
    union = A | B

    return len(interseccion) / len(union)

In [6]:
# Hacemos las 3 comparaciones,una por cada valor de epsilon en DBSCAN  ε ∈ {0.20, 0.30, 0.40}, vs Distance based (p=0.10, r=18)
epsilons = [0.20, 0.30, 0.40]

resultados_jaccard = []

for eps in epsilons:

    distance_mask = comparacion["anomalia_distance"].to_numpy()

    dbscan_mask = comparacion[
        f"ruido_eps_{eps:.2f}"
    ].to_numpy()

    A = set(np.where(distance_mask)[0])
    B = set(np.where(dbscan_mask)[0])

    interseccion = A & B
    union = A | B

    resultados_jaccard.append({
        "epsilon": eps,
        "anomalias_distance": len(A),
        "ruido_dbscan": len(B),
        "coincidencias": len(interseccion),
        "union": len(union),
        "jaccard": len(interseccion) / len(union)
    })

tabla_jaccard = pd.DataFrame(resultados_jaccard)

tabla_jaccard

,epsilon,anomalias_distance,ruido_dbscan,coincidencias,union,jaccard
0,0.2,359,76,62,373,0.166220
1,0.3,359,36,31,364,0.085165
2,0.4,359,6,5,360,0.013889
